NOTE : Since I'm using the Collab for this project, I couldn't create my own env, and the token for API expires/resets after one use. So the code contains the implementation with both huggingface and openAI

In [ ]:
import os
import io
import getpass

import requests                          # HTTP requests to call the HF Inference API
from PIL import Image                    # Python Imaging Library — image processing
from io import BytesIO                   # Manage binary data streams in memory
import gradio as gr                      # Gradio — web interface for ML models
from huggingface_hub import InferenceClient  # Hugging Face Inference API client
from openai import OpenAI             # OpenAI Python SDK


In [ ]:
# Set Hugging Face token from https://huggingface.co/settings/tokens
if "HF_TOKEN" not in os.environ:
    os.environ["HF_TOKEN"] = getpass.getpass("Enter your Hugging Face token (hf_...): ")

# Initialise the Hugging Face Inference client
client = InferenceClient(token=os.environ["HF_TOKEN"])

print("All libraries imported and Hugging Face client ready.")

In [ ]:
#Set OpenAI API key securely
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

# Option B: set directly (use only in private environments)
# os.environ["OPENAI_API_KEY"] = "sk-..."

# Initialise the OpenAI client
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

print("All libraries imported and OpenAI client ready.")

Use this for selecting the model from Hugging Face

In [ ]:
#  Model selection from Hugging Face
# Primary model: Stable Diffusion XL — high quality, 1024x1024 native
MODEL_ID = "stabilityai/stable-diffusion-xl-base-1.0"

# MODEL_ID = "runwayml/stable-diffusion-v1-5"         # Faster, lighter
# MODEL_ID = "stabilityai/stable-diffusion-2-1"       # Good all-rounder
# MODEL_ID = "prompthero/openjourney-v4"              # Midjourney-style art


def generate_image(
    prompt: str,
    negative_prompt: str = "",
    style_preset: str = "Cinematic", # Photorealistic, Digital Art, Minimalist, Photorealistic, Vintage Poster
    guidance_scale: float = 7.5, # How strictly to follow the prompt (1–20; 7–9 recommended)
    num_steps: int = 30, # Diffusion steps; more = higher quality but slower (20–50).
) -> Image.Image:

    if not prompt.strip():
        raise gr.Error("Please enter a text prompt before generating an image.")

    # Enrich the prompt with the chosen style preset
    style_keywords = {
        "Cinematic":       "cinematic lighting, film grain, dramatic shadows, 4K",
        "Digital Art":     "digital art, concept art, trending on ArtStation, vibrant",
        "Photorealistic":  "photorealistic, ultra-detailed, sharp focus, 8K DSLR photo",
        "Watercolor":      "watercolor painting, soft brush strokes, pastel tones",
        "Minimalist":      "minimalist design, clean composition, bold typography space",
        "Vintage Poster":  "vintage movie poster, retro illustration, aged paper texture",
    }
    enriched_prompt = f"{prompt.strip()}, {style_keywords.get(style_preset, '')}"

    #  Build a sensible default negative prompt
    default_negative = (
        "blurry, out of focus, low quality, distorted, watermark, "
        "text overlay, ugly, deformed, noisy, pixelated"
    )
    full_negative = f"{default_negative}, {negative_prompt}".strip(", ")

    try:
        # Call Hugging Face Inference API
        image = client.text_to_image(
            prompt=enriched_prompt,
            negative_prompt=full_negative,
            model=MODEL_ID,
            guidance_scale=guidance_scale,
            num_inference_steps=num_steps,
        )
        # The HF InferenceClient returns a PIL Image directly
        return image

    except Exception as exc:
        raise gr.Error(f"Image generation failed: {exc}")


#print("generate_image() function defined.")
#print(f"   Using model: {MODEL_ID}")

Use this block of code for selecting the model from Open AI

In [ ]:
def generate_image(
    prompt: str,
    size: str = "1024x1024",
    style: str = "vivid",
    quality: str = "standard"
) -> Image.Image:

    if not prompt.strip():
        raise gr.Error("Please enter a text prompt before generating an image.")

    try:
        # Call DALL-E 3 via the OpenAI API
        response = client.images.generate(
            model="dall-e-3",
            prompt=prompt,
            size=size,
            style=style,
            quality=quality,
            n=1                        # DALL-E 3 supports n=1 only
        )

        # Extract the image URL returned by the API
        image_url = response.data[0].url

        # Fetch image bytes via HTTP and convert to PIL Image
        image_response = requests.get(image_url, timeout=30)
        image_response.raise_for_status()

        image = Image.open(BytesIO(image_response.content)).convert("RGB")
        return image

    except Exception as exc:
        raise gr.Error(f"Image generation failed: {exc}")


#print(" generate_image() function defined.")

In [ ]:
demo = gr.Interface(
    fn=generate_image,
    inputs=gr.Textbox(
        label="Prompt",
        placeholder="e.g. A Netflix thriller poster with dramatic red lighting…",
        lines=3,
    ),
    outputs=gr.Image(
        label="Generated Image",
        show_download_button=True,
    ),
    title="Netflix Campaign Design Generator",
    description="Enter a text prompt to generate a banner or poster design.",
    examples=[
        ["A Netflix thriller poster, dark alleyway, silhouette, dramatic red tones"],
        ["Netflix fantasy banner, dragon over a medieval kingdom at sunset"],
        ["Minimalist Netflix horror poster, single candle in darkness"],
        ["Netflix documentary banner, bioluminescent ocean life, deep blue"],
    ],
    flagging_mode="never",
)

demo.launch(inbrowser=True)